# Signal Inversion Investigation

## Why Does the Manifold Contract During Crises?

### The Finding

Online detection diagnosis (`experiments/outputs/online_diagnosis/diagnosis_results.json`) revealed a striking
pattern: **all four geometric observables have LOWER mean values during crisis periods** than during normal periods:

| Observable | Crisis Mean | Normal Mean | Mann-Whitney p | Effect Size (d) |
|---|---|---|---|---|
| Berry curvature rate | 0.0009 | 0.0018 | 1.00 | 0.31 |
| QFI log-determinant | -37.63 | -35.45 | 1.00 | 1.79 |
| Infidelity (lag-1) | 0.0003 | 0.0005 | 0.96 | 0.07 |
| Inverse spectral gap | 0.213 | 0.309 | 1.00 | 1.48 |

This is why naive online detection (higher = crisis) achieves AUC-ROC ~ 0.10-0.38 (worse than random).
Simply negating the signals jumps AUC to ~0.84, but the deeper question is unexplored.

### Why This Matters

1. **Detection**: Understanding the direction enables better observable design.
2. **Understanding**: The manifold *contracting* during crises is geometrically meaningful &mdash;
   it implies that crisis states are *more similar to each other* than normal states.
3. **Observable design**: Rate-of-contraction or contraction-asymmetry may be naturally upward-pointing crisis signals.

### What We Investigate

1. **Direction**: Does every observable always go down? Or do some go up for some crises?
2. **Magnitude**: How large is the contraction (signed Cohen's d per crisis)?
3. **Timing**: Is the contraction gradual or sudden? Does it lead or lag the crisis?
4. **Per-crisis variation**: Do different crisis types produce different geometric signatures?
5. **Rate of change**: Does |d(signal)/dt| go UP during crises (naturally upward signal)?
6. **Contraction asymmetry**: Does the metric tensor collapse along specific eigenvalue directions?

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
from pathlib import Path
import warnings
import json
import logging

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from qcml_geometry.core import QCMLGeometry
from qcml_geometry.observables import BaseRegimeDetector
from experiments.data_loader import (
    fetch_data, create_feature_matrix_single_asset, ALL_CRISES, CRISIS_CATEGORIES,
)

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

ROOT = Path('..').resolve()
OUTPUT_DIR = ROOT / 'experiments' / 'outputs' / 'signal_inversion'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})

# Representative crises for detailed investigation
REPRESENTATIVE_CRISES = {
    '2008_gfc':            {'label': 'GFC 2008',          'type': 'conventional, prolonged, severe'},
    '2018_volmageddon':    {'label': 'Volmageddon 2018',  'type': 'novel, sharp, volatility-specific'},
    '2020_covid':          {'label': 'COVID 2020',        'type': 'conventional(?), extreme speed'},
    '2010_flash':          {'label': 'Flash Crash 2010',  'type': 'conventional, ultra-short'},
}

OBSERVABLE_NAMES = ['berry_rate', 'qfi_logdet', 'fidelity_lag5', 'spectral_gap']
OBSERVABLE_LABELS = {
    'berry_rate': 'Berry Curvature Rate',
    'qfi_logdet': 'QFI Log-Determinant',
    'fidelity_lag5': 'Multi-Lag Fidelity (lag=5)',
    'spectral_gap': 'Spectral Gap',
}

print('Setup complete.')

---
## Task 2: Compute Raw Observable Time Series

Load SPY (2000-2024) and compute raw, pre-z-score geometric observables at each timestep.

Config: `n_hilbert=8, n_pca=15, window=20, operator_method='random'` (Paper 1 defaults).

In [ ]:
# --- Data loading ---
print('Fetching SPY data 2000-2024...')
raw = fetch_data(['SPY'], '2000-01-01', '2024-12-31')
close = raw['close'].unstack('symbol').dropna()
print(f'Close prices: {close.shape[0]} days, {close.columns.tolist()}')

# Single-asset feature matrix with extra lags for sufficient dimensionality
X_raw, dates_raw = create_feature_matrix_single_asset(close['SPY'], extra_lags=True)
print(f'Raw features: {X_raw.shape}, dates: {dates_raw[0].date()} to {dates_raw[-1].date()}')

# Enrich with rolling statistics
LOOKBACK = 20
X_enriched = BaseRegimeDetector.build_enriched_features(X_raw, lookback=LOOKBACK)
dates = dates_raw[LOOKBACK - 1:]
T = len(X_enriched)
print(f'Enriched features: {X_enriched.shape}, T={T}')

In [ ]:
# --- Pipeline: scaler -> PCA -> normalize -> operators -> compute observables ---

N_HILBERT = 8
N_PCA = 15
REFIT_INTERVAL = 63
MIN_HISTORY = 126

# Storage for raw observables
raw_signals = {
    'berry_rate': np.full(T, np.nan),
    'qfi_logdet': np.full(T, np.nan),
    'fidelity_lag1': np.full(T, np.nan),
    'fidelity_lag5': np.full(T, np.nan),
    'fidelity_lag10': np.full(T, np.nan),
    'spectral_gap': np.full(T, np.nan),
}

# Also store metric eigenvalues for Task 6
metric_eigenvalues = [None] * T

prev_berry = None
prev_states = {}  # lag -> state
last_refit = 0
median_norm = 1.0
scaler = None
pca = None
geo = None

print(f'Computing observables for T={T} timesteps...')
print(f'Config: n_hilbert={N_HILBERT}, n_pca={N_PCA}, refit_interval={REFIT_INTERVAL}')

for t in range(MIN_HISTORY, T):
    if t % 500 == 0:
        print(f'  t={t}/{T} ({100*t/T:.0f}%)')

    # Refit periodically on expanding window
    if scaler is None or (t - last_refit >= REFIT_INTERVAL):
        scaler = StandardScaler()
        scaler.fit(X_enriched[:t])
        pca = PCA(n_components=min(N_PCA, X_enriched.shape[1]))
        X_scaled = scaler.transform(X_enriched[:t])
        pca.fit(X_scaled)

        X_pca_fit = pca.transform(X_scaled)
        norms = np.linalg.norm(X_pca_fit, axis=1, keepdims=True)
        median_norm = float(np.median(norms))
        X_pca_fit = X_pca_fit / (norms + median_norm)

        geo = QCMLGeometry(
            n_features=X_pca_fit.shape[1], hilbert_dim=N_HILBERT
        )
        geo.fit_operators(X_pca_fit, method='random')
        last_refit = t

    # Transform current point
    x_scaled = scaler.transform(X_enriched[t:t + 1])
    x_pca = pca.transform(x_scaled).ravel()
    norm = np.linalg.norm(x_pca)
    x_pca = x_pca / (norm + median_norm)

    # 1. Berry curvature rate: |berry(t) - berry(t-1)|
    berry = geo.berry_curvature_2d(x_pca, indices=(0, 1))
    if prev_berry is not None:
        raw_signals['berry_rate'][t] = abs(berry - prev_berry)
    prev_berry = berry

    # 2. QFI log-determinant: log(det(g))
    g = geo.quantum_metric(x_pca)
    eigs = np.linalg.eigvalsh(g)
    pos = eigs[eigs > 1e-10]
    raw_signals['qfi_logdet'][t] = np.sum(np.log(pos)) if len(pos) > 0 else -40.0

    # Store metric eigenvalues for Task 6
    metric_eigenvalues[t] = eigs.copy()

    # 3. Multi-lag fidelity: 1 - |<psi(t)|psi(t-lag)>|^2
    state = geo.quasi_coherent_state(x_pca)
    for lag in [1, 5, 10]:
        lag_key = f'fidelity_lag{lag}'
        if t - lag in prev_states:
            overlap = np.abs(np.vdot(state, prev_states[t - lag]))
            raw_signals[lag_key][t] = 1.0 - overlap ** 2
    prev_states[t] = state
    # Keep memory bounded - only need up to lag=10
    if t - 15 in prev_states:
        del prev_states[t - 15]

    # 4. Spectral gap: E_1 - E_0
    gap = geo.spectral_gap(x_pca)
    raw_signals['spectral_gap'][t] = gap

print(f'Done. Valid points per signal:')
for name, arr in raw_signals.items():
    n_valid = np.sum(~np.isnan(arr))
    print(f'  {name}: {n_valid}/{T} ({100*n_valid/T:.0f}%)')

In [ ]:
# --- Persist to parquet ---
signals_df = pd.DataFrame(raw_signals, index=dates)
signals_df.index.name = 'date'

parquet_path = OUTPUT_DIR / 'raw_signals.parquet'
signals_df.to_parquet(parquet_path)
print(f'Saved raw signals to {parquet_path}')
print(f'Shape: {signals_df.shape}')
signals_df.describe()

---
## Task 3: Visualize Signal Direction During Crises

For each of the 4 representative crises, plot the raw observable values around the crisis window.
Does the signal go UP or DOWN when entering the crisis? Gradual or sudden?

In [ ]:
def plot_crisis_signals(signals_df, crisis_key, crisis_info, observables, context_days=60):
    """Multi-panel figure showing raw observables around a crisis window.

    Args:
        signals_df: DataFrame with raw signal columns, DatetimeIndex.
        crisis_key: Key into ALL_CRISES.
        crisis_info: Dict with 'start', 'end', 'label'.
        observables: List of column names to plot.
        context_days: Days before/after crisis to show.
    """
    cs = pd.Timestamp(crisis_info['start'])
    ce = pd.Timestamp(crisis_info['end'])

    # Context window
    win_start = cs - pd.Timedelta(days=context_days)
    win_end = ce + pd.Timedelta(days=context_days)
    mask = (signals_df.index >= win_start) & (signals_df.index <= win_end)
    window_df = signals_df.loc[mask]

    if len(window_df) == 0:
        print(f'No data for {crisis_key}')
        return None

    # Normal period: 252 trading days before the context window
    normal_end = cs - pd.Timedelta(days=context_days + 1)
    normal_start = normal_end - pd.Timedelta(days=400)  # ~252 trading days
    normal_mask = (signals_df.index >= normal_start) & (signals_df.index <= normal_end)

    n_obs = len(observables)
    fig, axes = plt.subplots(n_obs, 1, figsize=(12, 3 * n_obs), sharex=True)
    if n_obs == 1:
        axes = [axes]

    for ax, obs_name in zip(axes, observables):
        vals = window_df[obs_name].dropna()
        if len(vals) == 0:
            continue

        # Raw signal
        ax.plot(vals.index, vals.values, linewidth=0.8, color='#1f77b4', alpha=0.6, label='Raw')

        # Rolling 20-day smoothed overlay
        smoothed = vals.rolling(20, min_periods=5).mean()
        ax.plot(smoothed.index, smoothed.values, linewidth=2.0, color='#1f77b4', label='20d smoothed')

        # Crisis window shading
        ax.axvspan(cs, ce, alpha=0.20, color='#d62728', label='Crisis window')

        # Pre-crisis shading (30 days before)
        pre_start = cs - pd.Timedelta(days=30)
        ax.axvspan(pre_start, cs, alpha=0.10, color='#ff7f0e', label='Pre-crisis (30d)')

        # Mean lines
        crisis_mask = (vals.index >= cs) & (vals.index <= ce)
        crisis_vals = vals[crisis_mask]
        normal_vals = signals_df.loc[normal_mask, obs_name].dropna()

        if len(crisis_vals) > 0:
            ax.axhline(crisis_vals.mean(), color='#d62728', linestyle='--', linewidth=1.0,
                      label=f'Crisis mean: {crisis_vals.mean():.4g}')
        if len(normal_vals) > 0:
            ax.axhline(normal_vals.mean(), color='#2ca02c', linestyle='--', linewidth=1.0,
                      label=f'Normal mean: {normal_vals.mean():.4g}')

        ax.set_ylabel(OBSERVABLE_LABELS.get(obs_name, obs_name), fontsize=9)
        ax.legend(fontsize=7, loc='upper right', ncol=2)
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))

    axes[-1].set_xlabel('Date')
    crisis_label = REPRESENTATIVE_CRISES.get(crisis_key, {}).get('label', crisis_key)
    crisis_type = REPRESENTATIVE_CRISES.get(crisis_key, {}).get('type', '')
    fig.suptitle(f'Signal Direction: {crisis_label}\n({crisis_type})', fontsize=12, y=1.02)
    fig.tight_layout()

    fig.savefig(OUTPUT_DIR / f'signal_direction_{crisis_key}.png')
    fig.savefig(OUTPUT_DIR / f'signal_direction_{crisis_key}.pdf')
    plt.show()
    return fig

In [ ]:
# Plot for each representative crisis
plot_observables = ['berry_rate', 'qfi_logdet', 'fidelity_lag5', 'spectral_gap']

for crisis_key in REPRESENTATIVE_CRISES:
    crisis_info = ALL_CRISES[crisis_key]
    print(f'\n=== {crisis_info["label"]} ===')
    plot_crisis_signals(signals_df, crisis_key, crisis_info, plot_observables)

---
## Task 4: Quantify Inversion Per Observable Per Crisis

For every (observable, crisis) pair, compute:
- **Direction**: sign(crisis_mean - normal_mean)
- **Magnitude**: signed Cohen's d
- **Significance**: Mann-Whitney U p-value
- **Timing**: onset lag (days to first 1-sigma deviation), peak lag (days to max deviation)

In [ ]:
def cohens_d(group1, group2):
    """Signed Cohen's d: positive means group1 > group2."""
    n1, n2 = len(group1), len(group2)
    if n1 < 2 or n2 < 2:
        return np.nan
    var1, var2 = np.var(group1, ddof=1), np.var(group2, ddof=1)
    pooled_std = np.sqrt(((n1 - 1) * var1 + (n2 - 1) * var2) / (n1 + n2 - 2))
    if pooled_std < 1e-15:
        return 0.0
    return (np.mean(group1) - np.mean(group2)) / pooled_std


def compute_onset_lag(signal_series, crisis_start, normal_mean, normal_std):
    """Days from crisis start to first sustained 1-sigma deviation."""
    cs = pd.Timestamp(crisis_start)
    post_crisis = signal_series[signal_series.index >= cs].dropna()
    if len(post_crisis) == 0 or normal_std < 1e-15:
        return np.nan
    threshold = normal_mean - normal_std  # 1-sigma below
    deviations = post_crisis < threshold
    if deviations.any():
        first_dev_date = deviations.idxmax()
        return (first_dev_date - cs).days
    return np.nan


def compute_peak_lag(signal_series, crisis_start, normal_mean):
    """Days from crisis start to maximum deviation from normal mean."""
    cs = pd.Timestamp(crisis_start)
    crisis_end_ext = cs + pd.Timedelta(days=180)
    post_crisis = signal_series[(signal_series.index >= cs) &
                                (signal_series.index <= crisis_end_ext)].dropna()
    if len(post_crisis) == 0:
        return np.nan
    deviations = np.abs(post_crisis.values - normal_mean)
    peak_idx = np.argmax(deviations)
    peak_date = post_crisis.index[peak_idx]
    return (peak_date - cs).days


# Compute summary table
results_rows = []

for crisis_key, crisis_info in ALL_CRISES.items():
    cs = pd.Timestamp(crisis_info['start'])
    ce = pd.Timestamp(crisis_info['end'])

    # Normal: 252 trading days before crisis (with 30-day buffer)
    normal_end = cs - pd.Timedelta(days=30)
    normal_start = normal_end - pd.Timedelta(days=400)
    normal_mask = (signals_df.index >= normal_start) & (signals_df.index <= normal_end)
    crisis_mask = (signals_df.index >= cs) & (signals_df.index <= ce)

    for obs_name in plot_observables:
        normal_vals = signals_df.loc[normal_mask, obs_name].dropna().values
        crisis_vals = signals_df.loc[crisis_mask, obs_name].dropna().values

        if len(normal_vals) < 10 or len(crisis_vals) < 3:
            continue

        direction = np.sign(np.mean(crisis_vals) - np.mean(normal_vals))
        d = cohens_d(crisis_vals, normal_vals)
        _, mw_p = stats.mannwhitneyu(crisis_vals, normal_vals, alternative='two-sided')

        normal_mean = np.mean(normal_vals)
        normal_std = np.std(normal_vals, ddof=1)
        onset = compute_onset_lag(signals_df[obs_name], crisis_info['start'],
                                  normal_mean, normal_std)
        peak = compute_peak_lag(signals_df[obs_name], crisis_info['start'], normal_mean)

        category = 'novel' if crisis_key in CRISIS_CATEGORIES.get('novel', []) else 'conventional'

        results_rows.append({
            'crisis': crisis_key,
            'crisis_label': crisis_info['label'],
            'category': category,
            'observable': obs_name,
            'direction': int(direction),
            'cohens_d': d,
            'mann_whitney_p': mw_p,
            'crisis_mean': np.mean(crisis_vals),
            'normal_mean': np.mean(normal_vals),
            'onset_lag_days': onset,
            'peak_lag_days': peak,
            'n_crisis': len(crisis_vals),
            'n_normal': len(normal_vals),
        })

results_df = pd.DataFrame(results_rows)
print(f'Computed {len(results_df)} (observable x crisis) measurements')
results_df.to_csv(OUTPUT_DIR / 'inversion_summary.csv', index=False)
results_df.head(10)

In [ ]:
# --- Direction summary: how often does each observable invert? ---
print('\n=== DIRECTION SUMMARY ===')
print('(direction = -1 means crisis < normal, i.e. inverted)\n')

for obs in plot_observables:
    obs_df = results_df[results_df['observable'] == obs]
    n_down = (obs_df['direction'] == -1).sum()
    n_up = (obs_df['direction'] == 1).sum()
    n_total = len(obs_df)
    mean_d = obs_df['cohens_d'].mean()
    print(f'{OBSERVABLE_LABELS[obs]:30s}  DOWN: {n_down}/{n_total}  UP: {n_up}/{n_total}  '
          f'mean d: {mean_d:+.3f}')

In [ ]:
# --- Heatmap of signed Cohen's d (observable x crisis) ---
pivot_d = results_df.pivot_table(
    index='observable', columns='crisis_label', values='cohens_d'
)

# Reorder observables to match plot_observables order (not alphabetical)
obs_order = [o for o in plot_observables if o in pivot_d.index]
plot_data = pivot_d.reindex(obs_order)

fig, ax = plt.subplots(figsize=(16, 4))
sns.heatmap(
    plot_data,
    annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    linewidths=0.5, ax=ax, vmin=-3, vmax=3,
    cbar_kws={'label': 'Signed Cohen\'s d (red = crisis > normal, blue = crisis < normal)'},
)
ax.set_title('Signal Direction: Signed Cohen\'s d per Observable x Crisis')
ax.set_ylabel('')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'cohens_d_heatmap.png')
fig.savefig(OUTPUT_DIR / 'cohens_d_heatmap.pdf')
plt.show()

In [ ]:
# --- Conventional vs Novel comparison ---
print('\n=== CONVENTIONAL vs NOVEL CRISES ===')
for obs in plot_observables:
    obs_df = results_df[results_df['observable'] == obs]
    conv = obs_df[obs_df['category'] == 'conventional']['cohens_d']
    novel = obs_df[obs_df['category'] == 'novel']['cohens_d']
    print(f'\n{OBSERVABLE_LABELS[obs]}:')
    print(f'  Conventional: mean d = {conv.mean():+.3f} (n={len(conv)}, '
          f'range [{conv.min():+.3f}, {conv.max():+.3f}])')
    print(f'  Novel:        mean d = {novel.mean():+.3f} (n={len(novel)}, '
          f'range [{novel.min():+.3f}, {novel.max():+.3f}])')

---
## Task 5: Rate-of-Change Observables

The raw signals *decrease* during crises. But does the **rate of decrease** increase?
This would be a naturally "upward-pointing" crisis signal.

Compute:
- First derivative: `d(signal)/dt`
- Second derivative: `d^2(signal)/dt^2` (acceleration)
- Absolute rate: `|d(signal)/dt|`

In [ ]:
# Compute derivatives for each observable
derivative_signals = {}

for obs_name in plot_observables:
    series = signals_df[obs_name].copy()

    # First derivative (forward difference)
    d1 = series.diff()
    derivative_signals[f'{obs_name}_d1'] = d1

    # Second derivative
    d2 = d1.diff()
    derivative_signals[f'{obs_name}_d2'] = d2

    # Absolute rate of change
    derivative_signals[f'{obs_name}_abs_d1'] = d1.abs()

    # Absolute acceleration
    derivative_signals[f'{obs_name}_abs_d2'] = d2.abs()

deriv_df = pd.DataFrame(derivative_signals, index=signals_df.index)
print(f'Derivative signals: {deriv_df.shape}')

In [ ]:
# Evaluate: does the derivative go UP during crises?
deriv_results = []

for crisis_key, crisis_info in ALL_CRISES.items():
    cs = pd.Timestamp(crisis_info['start'])
    ce = pd.Timestamp(crisis_info['end'])

    normal_end = cs - pd.Timedelta(days=30)
    normal_start = normal_end - pd.Timedelta(days=400)
    normal_mask = (deriv_df.index >= normal_start) & (deriv_df.index <= normal_end)
    crisis_mask = (deriv_df.index >= cs) & (deriv_df.index <= ce)

    for col in deriv_df.columns:
        normal_vals = deriv_df.loc[normal_mask, col].dropna().values
        crisis_vals = deriv_df.loc[crisis_mask, col].dropna().values

        if len(normal_vals) < 10 or len(crisis_vals) < 3:
            continue

        d = cohens_d(crisis_vals, normal_vals)
        direction = np.sign(np.mean(crisis_vals) - np.mean(normal_vals))

        # Parse the observable and derivative type
        if col.endswith('_abs_d2'):
            deriv_type = 'abs_d2'
            base_obs = col[:-7]
        elif col.endswith('_abs_d1'):
            deriv_type = 'abs_d1'
            base_obs = col[:-7]
        elif col.endswith('_d2'):
            deriv_type = 'd2'
            base_obs = col[:-3]
        elif col.endswith('_d1'):
            deriv_type = 'd1'
            base_obs = col[:-3]
        else:
            deriv_type = 'raw'
            base_obs = col

        deriv_results.append({
            'crisis': crisis_key,
            'observable': base_obs,
            'derivative': deriv_type,
            'direction': int(direction),
            'cohens_d': d,
        })

deriv_results_df = pd.DataFrame(deriv_results)
print(f'Computed {len(deriv_results_df)} derivative measurements')

In [ ]:
# --- Compare: raw d vs derivative d vs abs-derivative d ---
print('\n=== RAW vs DERIVATIVE: Mean Cohen\'s d Across All Crises ===')
print('(positive d = crisis HIGHER than normal = naturally upward)\n')
print(f'{"Observable":25s} {"raw":>8s} {"d1":>8s} {"abs_d1":>8s} {"d2":>8s} {"abs_d2":>8s}')
print('-' * 75)

for obs in plot_observables:
    # Raw signal d (from results_df)
    raw_d = results_df[results_df['observable'] == obs]['cohens_d'].mean()

    row = f'{OBSERVABLE_LABELS[obs]:25s} {raw_d:+8.3f}'
    for dt in ['d1', 'abs_d1', 'd2', 'abs_d2']:
        sub = deriv_results_df[
            (deriv_results_df['observable'] == obs) &
            (deriv_results_df['derivative'] == dt)
        ]
        mean_d = sub['cohens_d'].mean() if len(sub) > 0 else np.nan
        row += f' {mean_d:+8.3f}'
    print(row)

print('\nKey: positive values indicate the signal goes UP during crises (desirable).')
print('If abs_d1 is positive, the rate of change increases during crises.')

In [ ]:
# --- Visualize derivatives for 2008 GFC (most representative) ---
crisis_key = '2008_gfc'
crisis_info = ALL_CRISES[crisis_key]
cs = pd.Timestamp(crisis_info['start'])
ce = pd.Timestamp(crisis_info['end'])
context_days = 60

win_start = cs - pd.Timedelta(days=context_days)
win_end = ce + pd.Timedelta(days=context_days)
mask = (deriv_df.index >= win_start) & (deriv_df.index <= win_end)

fig, axes = plt.subplots(4, 2, figsize=(14, 12), sharex=True)

for i, obs in enumerate(plot_observables):
    # Left: raw signal
    ax_raw = axes[i, 0]
    raw_vals = signals_df.loc[mask, obs].dropna()
    ax_raw.plot(raw_vals.index, raw_vals.values, linewidth=0.8)
    ax_raw.axvspan(cs, ce, alpha=0.15, color='#d62728')
    ax_raw.set_ylabel(f'{OBSERVABLE_LABELS[obs]}\n(raw)', fontsize=8)

    # Right: absolute rate of change
    ax_deriv = axes[i, 1]
    abs_d1 = deriv_df.loc[mask, f'{obs}_abs_d1'].dropna()
    if len(abs_d1) > 0:
        ax_deriv.plot(abs_d1.index, abs_d1.values, linewidth=0.8, color='#ff7f0e')
        smoothed = abs_d1.rolling(20, min_periods=5).mean()
        ax_deriv.plot(smoothed.index, smoothed.values, linewidth=2.0, color='#d62728')
    ax_deriv.axvspan(cs, ce, alpha=0.15, color='#d62728')
    ax_deriv.set_ylabel(f'{OBSERVABLE_LABELS[obs]}\n(|d/dt|)', fontsize=8)

axes[0, 0].set_title('Raw Signal', fontsize=11)
axes[0, 1].set_title('Absolute Rate of Change', fontsize=11)
fig.suptitle(f'Raw vs Rate-of-Change: {crisis_info["label"]}', fontsize=13, y=1.02)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'derivatives_gfc.png')
fig.savefig(OUTPUT_DIR / 'derivatives_gfc.pdf')
plt.show()

---
## Task 6: Contraction Asymmetry

Does the manifold contract uniformly or along specific directions?

Using the quantum metric tensor eigenvalues:
- **IPR** (Inverse Participation Ratio): `max(eig) / sum(eig)` &mdash; 1/N if uniform, 1 if one direction dominates
- **Condition number**: `max(eig) / min(eig)` &mdash; shape of the ellipsoid
- **Trace**: `sum(eig)` &mdash; total manifold volume
- **Anisotropy**: `std(eig) / mean(eig)` &mdash; directional variation

In [ ]:
# Compute eigenvalue-based metrics from stored metric eigenvalues
eigen_metrics = {
    'ipr': np.full(T, np.nan),
    'condition': np.full(T, np.nan),
    'trace': np.full(T, np.nan),
    'anisotropy': np.full(T, np.nan),
    'max_eigenvalue': np.full(T, np.nan),
    'min_eigenvalue': np.full(T, np.nan),
}

for t in range(T):
    eigs = metric_eigenvalues[t]
    if eigs is None:
        continue

    pos_eigs = eigs[eigs > 1e-12]
    if len(pos_eigs) < 2:
        continue

    trace = np.sum(pos_eigs)
    eigen_metrics['ipr'][t] = np.max(pos_eigs) / trace
    eigen_metrics['condition'][t] = np.max(pos_eigs) / np.min(pos_eigs)
    eigen_metrics['trace'][t] = trace
    eigen_metrics['anisotropy'][t] = np.std(pos_eigs) / np.mean(pos_eigs)
    eigen_metrics['max_eigenvalue'][t] = np.max(pos_eigs)
    eigen_metrics['min_eigenvalue'][t] = np.min(pos_eigs)

eigen_df = pd.DataFrame(eigen_metrics, index=dates)
n_valid = eigen_df.notna().sum()
print('Eigenvalue metric coverage:')
for col in eigen_df.columns:
    print(f'  {col}: {n_valid[col]}/{T}')

In [ ]:
# --- Signed Cohen's d for eigenvalue metrics across all crises ---
eigen_results = []

for crisis_key, crisis_info in ALL_CRISES.items():
    cs = pd.Timestamp(crisis_info['start'])
    ce = pd.Timestamp(crisis_info['end'])

    normal_end = cs - pd.Timedelta(days=30)
    normal_start = normal_end - pd.Timedelta(days=400)
    normal_mask = (eigen_df.index >= normal_start) & (eigen_df.index <= normal_end)
    crisis_mask = (eigen_df.index >= cs) & (eigen_df.index <= ce)

    for col in eigen_df.columns:
        normal_vals = eigen_df.loc[normal_mask, col].dropna().values
        crisis_vals = eigen_df.loc[crisis_mask, col].dropna().values

        if len(normal_vals) < 10 or len(crisis_vals) < 3:
            continue

        d = cohens_d(crisis_vals, normal_vals)
        direction = np.sign(np.mean(crisis_vals) - np.mean(normal_vals))
        category = 'novel' if crisis_key in CRISIS_CATEGORIES.get('novel', []) else 'conventional'

        eigen_results.append({
            'crisis': crisis_key,
            'crisis_label': crisis_info['label'],
            'category': category,
            'metric': col,
            'direction': int(direction),
            'cohens_d': d,
        })

eigen_results_df = pd.DataFrame(eigen_results)

# Summary
print('\n=== EIGENVALUE METRICS: Mean Cohen\'s d Across All Crises ===')
for col in eigen_df.columns:
    sub = eigen_results_df[eigen_results_df['metric'] == col]
    n_up = (sub['direction'] == 1).sum()
    n_down = (sub['direction'] == -1).sum()
    mean_d = sub['cohens_d'].mean()
    print(f'  {col:20s}  d={mean_d:+.3f}  UP:{n_up}  DOWN:{n_down}')

In [ ]:
# --- Visualize eigenvalue metrics for representative crises ---
eigen_plot_cols = ['ipr', 'condition', 'trace', 'anisotropy']
eigen_labels = {
    'ipr': 'IPR (Dimensionality Collapse)',
    'condition': 'Condition Number',
    'trace': 'Trace (Total Volume)',
    'anisotropy': 'Anisotropy (std/mean)',
}

for crisis_key in ['2008_gfc', '2020_covid']:
    crisis_info = ALL_CRISES[crisis_key]
    cs = pd.Timestamp(crisis_info['start'])
    ce = pd.Timestamp(crisis_info['end'])
    context_days = 60

    win_start = cs - pd.Timedelta(days=context_days)
    win_end = ce + pd.Timedelta(days=context_days)
    mask = (eigen_df.index >= win_start) & (eigen_df.index <= win_end)

    fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

    for ax, col in zip(axes, eigen_plot_cols):
        vals = eigen_df.loc[mask, col].dropna()
        if len(vals) == 0:
            continue
        ax.plot(vals.index, vals.values, linewidth=0.8, color='#1f77b4', alpha=0.5)
        smoothed = vals.rolling(20, min_periods=5).mean()
        ax.plot(smoothed.index, smoothed.values, linewidth=2.0, color='#1f77b4')
        ax.axvspan(cs, ce, alpha=0.15, color='#d62728')
        ax.set_ylabel(eigen_labels[col], fontsize=9)

    axes[-1].set_xlabel('Date')
    fig.suptitle(f'Metric Eigenvalue Metrics: {crisis_info["label"]}', fontsize=12, y=1.02)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'eigenvalue_metrics_{crisis_key}.png')
    fig.savefig(OUTPUT_DIR / f'eigenvalue_metrics_{crisis_key}.pdf')
    plt.show()

In [ ]:
# --- Heatmap: eigenvalue metrics x crisis ---
pivot_eigen = eigen_results_df.pivot_table(
    index='metric', columns='crisis_label', values='cohens_d'
)

# Reorder to match eigen_df.columns order (not alphabetical)
eigen_order = [c for c in eigen_df.columns if c in pivot_eigen.index]
plot_data = pivot_eigen.reindex(eigen_order)

fig, ax = plt.subplots(figsize=(16, 5))
sns.heatmap(
    plot_data, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
    linewidths=0.5, ax=ax, vmin=-3, vmax=3,
    cbar_kws={'label': 'Signed Cohen\'s d'},
)
ax.set_title('Metric Eigenvalue Metrics: Signed Cohen\'s d per Crisis')
ax.set_ylabel('')
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'eigenvalue_heatmap.png')
fig.savefig(OUTPUT_DIR / 'eigenvalue_heatmap.pdf')
plt.show()

---
## Task 7: Synthesis

### Key Questions Answered
1. Which observables invert? Always or sometimes? By how much?
2. Do different crisis types produce different geometric signatures?
3. Do derivative observables provide naturally upward crisis signals?
4. Is the contraction isotropic or anisotropic?

In [ ]:
# === SYNTHESIS ===

print('=' * 80)
print('SIGNAL INVERSION INVESTIGATION: SYNTHESIS')
print('=' * 80)

# 1. Inversion universality
print('\n--- 1. INVERSION UNIVERSALITY ---')
print('How often does each observable decrease during crises?\n')
for obs in plot_observables:
    obs_df = results_df[results_df['observable'] == obs]
    n_down = (obs_df['direction'] == -1).sum()
    n_total = len(obs_df)
    mean_d = obs_df['cohens_d'].mean()
    sig_crises = obs_df[obs_df['mann_whitney_p'] < 0.05]
    n_sig = len(sig_crises)
    print(f'  {OBSERVABLE_LABELS[obs]:30s}  '
          f'DOWN: {n_down}/{n_total} ({100*n_down/n_total:.0f}%)  '
          f'mean d: {mean_d:+.3f}  '
          f'significant (p<0.05): {n_sig}/{n_total}')

# 2. Crisis taxonomy
print('\n--- 2. CRISIS TAXONOMY ---')
print('Mean signed d by crisis type:\n')
for cat in ['conventional', 'novel']:
    cat_df = results_df[results_df['category'] == cat]
    mean_d = cat_df['cohens_d'].mean()
    print(f'  {cat:15s}: mean d = {mean_d:+.3f} (n={len(cat_df)})')

# Per-crisis summary
print('\nPer-crisis mean d (across all observables):')
crisis_mean_d = results_df.groupby('crisis_label')['cohens_d'].mean().sort_values()
for label, d in crisis_mean_d.items():
    print(f'  {label:30s}: {d:+.3f}')

# 3. Derivative analysis
print('\n--- 3. DERIVATIVE ANALYSIS ---')
print('Does |d/dt| go UP during crises? (positive mean d = yes)\n')
for obs in plot_observables:
    raw_d = results_df[results_df['observable'] == obs]['cohens_d'].mean()
    abs_d1_data = deriv_results_df[
        (deriv_results_df['observable'] == obs) &
        (deriv_results_df['derivative'] == 'abs_d1')
    ]
    abs_d1_mean = abs_d1_data['cohens_d'].mean() if len(abs_d1_data) > 0 else np.nan
    n_up = (abs_d1_data['direction'] == 1).sum() if len(abs_d1_data) > 0 else 0
    n_total = len(abs_d1_data)
    improvement = abs_d1_mean - raw_d if not np.isnan(abs_d1_mean) else np.nan

    print(f'  {OBSERVABLE_LABELS[obs]:30s}  '
          f'raw: {raw_d:+.3f}  |d/dt|: {abs_d1_mean:+.3f}  '
          f'UP: {n_up}/{n_total}  '
          f'improvement: {improvement:+.3f}')

# 4. Contraction asymmetry
print('\n--- 4. CONTRACTION ASYMMETRY ---')
print('Eigenvalue metric behavior during crises:\n')
for col in eigen_plot_cols:
    sub = eigen_results_df[eigen_results_df['metric'] == col]
    mean_d = sub['cohens_d'].mean()
    n_up = (sub['direction'] == 1).sum()
    n_total = len(sub)
    print(f'  {eigen_labels[col]:35s}  d={mean_d:+.3f}  UP:{n_up}/{n_total}')

In [ ]:
# === CANDIDATE NEW OBSERVABLES ===

print('\n' + '=' * 80)
print('CANDIDATE NEW OBSERVABLES')
print('=' * 80)

candidates = []

# Check which derivatives show promise
for obs in plot_observables:
    for dt in ['abs_d1', 'abs_d2']:
        sub = deriv_results_df[
            (deriv_results_df['observable'] == obs) &
            (deriv_results_df['derivative'] == dt)
        ]
        if len(sub) == 0:
            continue
        mean_d = sub['cohens_d'].mean()
        n_up = (sub['direction'] == 1).sum()
        n_total = len(sub)
        if mean_d > 0 and n_up > n_total / 2:
            candidates.append({
                'name': f'{obs}_{dt}',
                'type': 'Rate of contraction',
                'mean_d': mean_d,
                'direction_consistency': n_up / n_total,
                'rationale': f'Absolute {"rate" if dt=="abs_d1" else "acceleration"} of {obs} '
                             f'increases during crises (d={mean_d:+.3f}, {n_up}/{n_total} UP)',
            })

# Check eigenvalue metrics
for col in eigen_df.columns:
    sub = eigen_results_df[eigen_results_df['metric'] == col]
    if len(sub) == 0:
        continue
    mean_d = sub['cohens_d'].mean()
    n_up = (sub['direction'] == 1).sum()
    n_total = len(sub)
    if abs(mean_d) > 0.3:
        candidates.append({
            'name': col,
            'type': 'Contraction asymmetry',
            'mean_d': mean_d,
            'direction_consistency': max(n_up, n_total - n_up) / n_total,
            'rationale': f'{col} shows d={mean_d:+.3f} during crises '
                         f'({"UP" if mean_d > 0 else "DOWN"} in '
                         f'{max(n_up, n_total-n_up)}/{n_total} crises)',
        })

# Also add negated raw signals as candidates
for obs in plot_observables:
    obs_df = results_df[results_df['observable'] == obs]
    mean_d = obs_df['cohens_d'].mean()
    if mean_d < -0.3:
        candidates.append({
            'name': f'neg_{obs}',
            'type': 'Signed (negated) observable',
            'mean_d': -mean_d,
            'direction_consistency': (obs_df['direction'] == -1).mean(),
            'rationale': f'Negated {obs}: reliable downward signal (d={mean_d:+.3f}), '
                         f'negation makes it upward',
        })

# Sort candidates by mean_d (higher is better for detection)
candidates_df = pd.DataFrame(candidates).sort_values('mean_d', ascending=False)

print(f'\nFound {len(candidates_df)} candidate observables:\n')
for _, row in candidates_df.iterrows():
    print(f'  [{row["type"]}] {row["name"]}')
    print(f'    mean d = {row["mean_d"]:+.3f}, consistency = {row["direction_consistency"]:.0%}')
    print(f'    {row["rationale"]}')
    print()

# Save
candidates_df.to_csv(OUTPUT_DIR / 'candidate_observables.csv', index=False)

In [ ]:
# === PAPER RECOMMENDATIONS ===

print('\n' + '=' * 80)
print('RECOMMENDATIONS')
print('=' * 80)

print('''
PAPER 1 (Geometric Observatory):
  - Document the signal inversion finding prominently in the Discussion
  - The finding that geometric observables CONTRACT during crises is itself
    a key contribution: crisis states are more geometrically similar than
    normal states, suggesting market dimensionality collapses
  - Add negated signals or direction-aware detection to the online results

PAPER 2 (Geometric Dynamics):
  - Rate-of-contraction observables (if promising from this analysis)
  - Eigenvalue-based crisis taxonomy (different crises = different contraction patterns)
  - Temporal derivative signals as naturally upward-pointing observables
  - Connection to information geometry: contraction = loss of distinguishability
''')

print('\nInvestigation complete. All outputs saved to:')
print(f'  {OUTPUT_DIR}')
for f in sorted(OUTPUT_DIR.iterdir()):
    if f.name != '.cache':
        print(f'    {f.name}')